In [97]:
import requests                      # To download webpage HTML
from bs4 import BeautifulSoup        # To parse HTML and extract text
import pandas as pd                  # To save data to CSV
import spacy                        # NLP model for keyword extraction
import nltk                        # For stopwords
from nltk.corpus import stopwords

# Download necessary NLTK data (run only once)
nltk.download("stopwords")
nltk.download("punkt")

# Define English stopwords (common words to ignore)
stop_words = set(stopwords.words("english"))

# Load spaCy English language model
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marut\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\marut\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [11]:
pip install python-docx

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 20.6 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [lxml]
   -------------------- ------------------- 1/2 [python-docx]
   ---------------------------------------- 2/2 [python-docx]

Note: you may need to restart the kernel to use updated packages.


In [99]:
import os
import re
import csv
import docx

def extract_resume_info(docx_path):
    doc = docx.Document(docx_path)
    text = "\n".join([para.text for para in doc.paragraphs])

    # Extract name
    def extract_name(text):
        lines = text.splitlines()
        for i, line in enumerate(lines):
            if "resume:" in line.lower():
                # take the next non-empty line as name
                for next_line in lines[i+1:]:
                    next_line = next_line.strip()
                    if next_line:
                        return next_line
        return None

    name = extract_name(text) or "N/A"  # make sure name is set

    # Extract email
    email_match = re.search(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", text)
    email = email_match.group(0) if email_match else "N/A"

    # Extract phone number
    phone_match = re.search(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{3,4}\b", text)
    phone = phone_match.group(0) if phone_match else "N/A"

    # Extract education info
    education_matches = re.findall(r"(Bachelor|Master|Certificate|Diploma|Ph\.?D\.?)\s+(?:in)?\s+([A-Za-z\s&\-]+)", text, re.IGNORECASE)
    educations = [{"level": level.strip(), "course": course.strip()} for level, course in education_matches]

    return {
        "Name": name,
        "Email": email,
        "Phone": phone,
        "Education": educations
    }

def save_to_csv(resume_info, csv_path="candidates_info.csv"):
    file_exists = os.path.isfile(csv_path)

    with open(csv_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(["Name", "Email", "Phone", "Education_Level", "Education_Course"])
        if resume_info["Education"]:
            for edu in resume_info["Education"]:
                writer.writerow([resume_info["Name"], resume_info["Email"], resume_info["Phone"], edu["level"], edu["course"]])
        else:
            writer.writerow([resume_info["Name"], resume_info["Email"], resume_info["Phone"], "N/A", "N/A"])

def process_resume(path):
    info = extract_resume_info(path)
    save_to_csv(info)
    print(f"✅ Info from {os.path.basename(path)} saved.")

# Run this with your resume path


In [103]:
resume_path = r"C:\Users\marut\Desktop\capstone\sample2.docx"
process_resume(resume_path)

✅ Info from sample2.docx saved.
